# Fashion MNIST Classification using Artificial Neural Network (ANN) in PyTorch
This notebook implements a complete Artificial Neural Network (ANN) to classify fashion items from the **Fashion MNIST** dataset.

We will perform:
1. Dataset loading from a local CSV and image grid visualization using `matplotlib`.
2. Preprocessing features by normalizing pixel intensities (dividing by 255).
3. Wrapping the image data inside a custom PyTorch `Dataset` and batching with `DataLoader`.
4. Constructing a Multi-Layer Perceptron (MLP) classification network.
5. Training the network with Cross-Entropy Loss and SGD optimizer.
6. Evaluating model performance on unseen test data.

## 1. Importing Libraries
We load Pandas for file reading, Scikit-learn for splitting, Matplotlib for graphics, and PyTorch packages for deep learning.

In [ ]:
# Core libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

## 2. Setting Reproducibility Seeds

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)

## 3. Loading the Dataset
The Fashion MNIST data consists of 28x28 grayscale images flattened into 784 pixel columns, with an initial label column.

In [ ]:
# Loading the local Fashion MNIST small dataset
df = pd.read_csv("fmnist_small.csv")
df.head()

## 4. Visualizing Sample Images
We reconstruct the 784 flattened pixel dimensions back to a 28x28 grid to inspect the clothing item images.

In [ ]:
# Create a 4x4 grid of images
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle("First 16 Fashion Items in Dataset", fontsize=16)

# Plot the first 16 images
for i, ax in enumerate(axes.flat):
    # Extract pixels, reshape from 784 to 28x28
    img = df.iloc[i, 1:].values.reshape(28, 28)  
    # Plot image in default colormap
    ax.imshow(img)
    ax.axis('off')  # Hide axis rulers
    ax.set_title(f"Label: {df.iloc[i, 0]}")  # Add label title

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 5. Splitting Training and Test Data

In [ ]:
# Separate features (X) and labels (Y)
x = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

# Train-test split (80-20 partition)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## 6. Normalization / Feature Scaling
We normalize the pixel intensities from range `[0, 255]` to `[0, 1]` to help model optimization converge faster.

In [ ]:
# Normalize pixel values by scaling by 255.0
x_train = x_train / 255.0
x_test = x_test / 255.0

## 7. Creating custom PyTorch Dataset
We inherit from PyTorch's `Dataset` class and cast features to Float32 tensors and target label categories to Long (64-bit integer) tensors, which are required for Cross-Entropy classification loss in PyTorch.

In [ ]:
# Custom Dataset class for Fashion MNIST
class CustomData(Dataset):
    def __init__(self, features, labels):
        # Features as float32; labels as long (required for CrossEntropyLoss)
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

## 8. Instantiating Dataset and DataLoaders

In [ ]:
# Create dataset wrappers
train_dataset = CustomData(x_train, y_train)
test_dataset = CustomData(x_test, y_test)

In [ ]:
# Create batch dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 9. Designing the ANN Architecture
We design a Multi-Layer Perceptron (MLP) consisting of:
- Input layer (784 inputs -> 128 nodes)
- Hidden layer 1 (128 nodes -> 64 nodes with ReLU activation)
- Output layer (64 nodes -> 10 output class logits)

In [ ]:
# Define Artificial Neural Network Class
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        # Multi-layer MLP architecture
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
            # Note: No Softmax layer because CrossEntropyLoss applies it internally
        )

    def forward(self, x):
        return self.model(x)

## 10. Training Hyperparameters

In [ ]:
# Configure epochs and optimization learning rate
epochs = 100
learning_rate = 0.1

## 11. Initializing Model, Loss Function, and Optimizer

In [ ]:
# Instantiate model
model = MyNN(x_train.shape[1])

# Define Cross-Entropy Loss
criterion = nn.CrossEntropyLoss()

# Define SGD optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

## 12. Executing the Training Loop
We run the optimization loop: feed batches -> compute forward prediction -> measure categorical cross entropy -> backpropagate gradients -> apply updates.

In [ ]:
# Start model training loop
for epoch in range(epochs):
    total_epoch_loss = 0.0
    
    for batch_features, batch_labels in train_loader:
        # 1. Forward Pass
        outputs = model(batch_features)
        
        # 2. Compute classification loss
        loss = criterion(outputs, batch_labels)
        
        # 3. Clear existing parameter gradients
        optimizer.zero_grad()
        
        # 4. Backward propagation
        loss.backward()
        
        # 5. Gradient updates
        optimizer.step()
        
        total_epoch_loss += loss.item()
        
    avg_loss = total_epoch_loss / len(train_loader)
    
    # Print training loss summary every 10 epochs for clean logs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch: {epoch + 1:03d} | Avg Loss: {avg_loss:.5f}')

## 13. Putting Model in Evaluation Mode
We toggle the model to `.eval()` mode, which disables operations like Dropout or Batch Normalization during inference.

In [ ]:
# Switch to evaluation mode
model.eval()

## 14. Calculating Accuracy on Test Set
We feed test features to our trained network, extract the indices of the largest output logits (using `torch.max`), and verify accuracy against ground truth targets.

In [ ]:
# Model Evaluation
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Feed forward test inputs
        outputs = model(batch_features)
        
        # Pick the class index with highest logit/probability score
        _, predicted = torch.max(outputs, dim=1)
        
        # Update sample counter and match accuracy count
        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()

accuracy = correct / total
print(f"Final Accuracy on test set: {accuracy * 100:.2f}%")

## 15. Verifying Test Loader Batch Count

In [ ]:
# Count total number of test batches
print("Total Test Batches:", len(test_loader))